# Training Demo

Train the **StrokeGAT** model for stroke lesion node classification.

- **Architecture:** 3-layer GAT, 8 heads, 128 hidden dim, PAA module (Eq. 8-12)
- **Loss:** Weighted cross-entropy + Dice (Eq. 13)
- **Optimizer:** Adam (lr=0.001, betas=(0.9, 0.999)), ExponentialLR (gamma=0.95)
- **Selection:** Composite score = 0.5*Dice + 0.3*AUC + 0.2*Sensitivity

> **Note:** This demo uses reduced epochs (20) for quick iteration.

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, "../src")

import torch
import numpy as np
import matplotlib.pyplot as plt

from stroke_gat.config import load_config
from stroke_gat.data.loader import StrokeDataModule
from stroke_gat.models.gat import StrokeGAT
from stroke_gat.training.trainer import Trainer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load config with reduced epochs for demo
config = load_config("../configs/default.yaml")
config.training.epochs = 20  # reduced for demo

print("Training configuration:")
print(f"  Epochs:          {config.training.epochs}")
print(f"  Batch size:      {config.training.batch_size}")
print(f"  Learning rate:   {config.training.lr}")
print(f"  LR decay rate:   {config.training.lr_decay_rate}")
print(f"  Weight decay:    {config.training.weight_decay}")
print(f"  Early stopping:  {config.training.early_stopping_patience}")
print(f"  Device:          {device}")

In [ ]:
# Setup data module (loads pre-computed .pt graph files)
data_module = StrokeDataModule(
    training_config=config.training,
    paths_config=config.paths,
)
data_module.setup()

print(f"Dataset splits:")
print(f"  Train: {len(data_module.train_dataset):>5} graphs")
print(f"  Val:   {len(data_module.val_dataset):>5} graphs")
print(f"  Test:  {len(data_module.test_dataset):>5} graphs")

# Peek at a sample
sample = data_module.train_dataset.get(0)
print(f"\nSample graph:")
print(f"  Nodes: {sample.num_nodes}, Edges: {sample.num_edges}")
print(f"  Feature dim: {sample.x.shape[1]}")
print(f"  Labels: {torch.bincount(sample.y).tolist()}")

In [ ]:
# Create the StrokeGAT model
in_dim = data_module.get_input_dim()
model = StrokeGAT(in_channels=in_dim, config=config.model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("StrokeGAT Architecture")
print("=" * 50)
print(model)
print("=" * 50)
print(f"Total parameters:     {total_params:>10,}")
print(f"Trainable parameters: {trainable_params:>10,}")

In [ ]:
# Create trainer and run training
trainer = Trainer(
    model=model,
    data_module=data_module,
    training_config=config.training,
    model_config=config.model,
    device=device,
    output_dir="../outputs",
)

print("Starting training...\n")
results = trainer.train()
history = results["history"]

print(f"\nTraining complete!")
print(f"Best composite score: {results['best_metrics'].get('composite_score', 0):.4f}")

In [ ]:
# Plot training curves
epochs = [h["epoch"] + 1 for h in history]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss
axes[0, 0].plot(epochs, [h["train_loss"] for h in history], "b-", label="Train", linewidth=2)
axes[0, 0].plot(epochs, [h["val_loss"] for h in history], "r-", label="Val", linewidth=2)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].set_title("Training & Validation Loss")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Composite Score
axes[0, 1].plot(epochs, [h["composite_score"] for h in history], "g-", linewidth=2)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Composite Score")
axes[0, 1].set_title("Composite Score (0.5*Dice + 0.3*AUC + 0.2*Sensitivity)")
axes[0, 1].grid(True, alpha=0.3)

# Dice Score
axes[1, 0].plot(epochs, [h["dice_mean"] for h in history], "m-", linewidth=2)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("Dice")
axes[1, 0].set_title("Mean Dice Coefficient")
axes[1, 0].grid(True, alpha=0.3)

# AUC
axes[1, 1].plot(epochs, [h["auc_roc"] for h in history], "c-", linewidth=2)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("AUC-ROC")
axes[1, 1].set_title("Area Under ROC Curve")
axes[1, 1].grid(True, alpha=0.3)

fig.suptitle("Training Curves", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set with best checkpoint
test_results = trainer.test()

print(f"{'Metric':<25} {'Value':>10}")
print("=" * 37)
print(f"{'Composite Score':<25} {test_results.get('composite_score', 0.0):>10.4f}")
print(f"{'Dice Mean':<25} {test_results.get('dice_mean', 0.0):>10.4f}")
print(f"{'AUC-ROC':<25} {test_results.get('auc_roc', 0.0):>10.4f}")
print(f"{'Balanced Accuracy':<25} {test_results.get('balanced_accuracy', 0.0):>10.4f}")
print(f"{'Sensitivity (macro)':<25} {test_results.get('sensitivity_macro', 0.0):>10.4f}")
print("-" * 37)

from stroke_gat.training.metrics import MetricsComputer
for name in MetricsComputer.CLASS_NAMES:
    print(f"{name:12s}  dice={test_results.get(f'{name}_dice', 0):.4f}  "
          f"sens={test_results.get(f'{name}_sensitivity', 0):.4f}  "
          f"spec={test_results.get(f'{name}_specificity', 0):.4f}")

## Discussion

**Full training (200 epochs) paper targets:** Dice ~0.85 with 5-fold cross-validation.

**Hyperparameter tips:**
1. LR: Start with 0.001, ExponentialLR (gamma=0.95) handles decay
2. Heads: 8 heads work well for this task
3. Hidden dim: 128 balances capacity and speed
4. Dropout: 0.1 default; increase to 0.2-0.3 if overfitting
5. Class weights: Computed automatically from training label distribution

Next: See `05_attention_attribution_gallery.ipynb` for model interpretability.